# Chuẩn bị môi trường

In [1]:
import os
import csv
import cv2
import mediapipe as mp
from tqdm import tqdm
import matplotlib.pyplot as plt


# Tải dataset

In [2]:
DATASET_DIR = "../../data/emotion"          
OUTPUT_CSV = "../../Output/face_landmarks.csv"
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")

mp_face_mesh = mp.solutions.face_mesh

NUM_LANDMARKS = 478

# Đánh số 478 toạ độ

In [3]:
def danhsotoado():
    header = []
    header.append("filepath")
    for i in range(NUM_LANDMARKS):
        header += [f"x{i}", f"y{i}", f"z{i}"]
    header.append("label")
    return header

# Trích xuất 478 toạ độ từ ảnh

In [4]:
def trich_xuat_toa_do_tu_anh(face_mesh, image_path):
    """Trả về list 1434 giá trị (x,y,z * 478) hoặc None nếu không detect được mặt."""
    image = cv2.imread(image_path)
    if image is None:
        return None

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) 
    # CV2 đọc ảnh theo Blue-Green-Red, nhưng mà Mediapipe đọc theo Red-Green-Blue dẫn đến quá trình trích xuất sai lệch.
    # Do vậy cần phải convert ảnh từ BGR sang RGB để trích xuất đúng.
    results = face_mesh.process(image_rgb)

    if not results.multi_face_landmarks: # Không tìm được mặt thì return None
        return None

    face_landmarks = results.multi_face_landmarks[0] # Nếu detect từ 2 mặt trở lên thì lấy cái mặt được detect đầu tiên để gắn toạ độ

    row = []
    for lm in face_landmarks.landmark:
        row.extend([lm.x, lm.y, lm.z]) # Lấy toạ độ (x,y,z) của 478 điểm

    return row

# Gắn nhãn cho mỗi ảnh

In [5]:
def load_label_id(class_id_path):
    """
    Đọc file Class_ID.txt dạng:
        0: angry
        1: disgust
        2: fear
        ...
    Trả về dict: {"angry": 0, "disgust": 1, "fear": 2, ...}
    """
    name_to_id = {}
    with open(class_id_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()          # bỏ khoảng trắng/xuống dòng thừa
            if not line:
                continue                 # bỏ qua dòng trống
            id_str, name = line.split(":", 1)   # tách theo dấu ":"
            class_id = int(id_str.strip())      # "0" → 0 (số nguyên)
            class_name = name.strip()           # " angry" → "angry"
            name_to_id[class_name] = class_id

    return name_to_id

In [ ]:
header = danhsotoado()
class_mapping = load_label_id("../../docs/Class_ID.txt")
rows_written = 0
rows_skipped = 0
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
with mp_face_mesh.FaceMesh(
    static_image_mode=True,      # bắt buộc True vì xử lý ảnh tĩnh, không phải xử lí theo thời gian thực
    max_num_faces=1,
    refine_landmarks=True,       # bật thêm landmark quanh mắt/môi cho chính xác hơn
    min_detection_confidence=0.5,
) as face_mesh, open(OUTPUT_CSV, "w", newline="") as f_out:

    writer = csv.writer(f_out)
    writer.writerow(header) # Tạo hàng tiêu đề cho file CSV

    labels = sorted(
        d for d in os.listdir(DATASET_DIR)
        if os.path.isdir(os.path.join(DATASET_DIR, d))
        if d.lower() in class_mapping
    )
    print(f"Tìm thấy {len(labels)} nhãn: {labels}")
    scan_fail =[]
    scan_success =[]
    for label in labels:
        label_dir = os.path.join(DATASET_DIR, label)
        image_files = [
            fn for fn in os.listdir(label_dir)
            if fn.lower().endswith(IMAGE_EXTENSIONS)
        ]
        fail =0
        success =0
        for fn in tqdm(image_files, desc=f"Đang xử lý '{label}'"):
            image_path = os.path.join(label_dir, fn)
            landmarks = trich_xuat_toa_do_tu_anh(face_mesh, image_path)

            if landmarks is None:
                rows_skipped += 1
                fail+=1
                continue
            label_id = class_mapping.get(label, -1)  # mặc định -1 nếu không có trong mapping
            relative_path = os.path.join(label, fn)
            writer.writerow([relative_path] + landmarks + [label_id])
            rows_written += 1
            success +=1
        scan_fail.append(fail)
        scan_success.append(success)
print(f"\nXong! Ghi được {rows_written} dòng vào '{OUTPUT_CSV}'.")
print(f"Bỏ qua {rows_skipped} ảnh (không detect được mặt).")

Tìm thấy 5 nhãn: ['angry', 'happy', 'neutral', 'sad', 'surprise']


Đang xử lý 'surprise':   2%|▏         | 99/4386 [00:02<01:50, 38.91it/s]


Xong! Ghi được 498 dòng vào '../../Output/face_landmarks.csv'.
Bỏ qua 2 ảnh (không detect được mặt).


In [7]:
import pandas as pd

df = pd.read_csv(OUTPUT_CSV)

print(df.head(1).shape)
display(df.head())

(1, 1436)


,filepath,x0,y0,z0,x1,y1,z1,x2,y2,z2,...,x475,y475,z475,x476,y476,z476,x477,y477,z477,label
0,angry\001_camA_S1_angry_00000.jpg,0.548771,0.720122,-0.047389,0.560323,0.630263,-0.077924,0.555654,0.659643,-0.043860,...,0.613478,0.466092,0.033263,0.597074,0.486576,0.033263,0.611829,0.511410,0.033263,0
1,angry\001_camA_S1_angry_00001.jpg,0.548189,0.723357,-0.046142,0.563565,0.631643,-0.077393,0.558372,0.661094,-0.043025,...,0.613898,0.465254,0.032801,0.599042,0.486800,0.032801,0.612888,0.512213,0.032801,0
2,angry\001_camA_S1_angry_00002.jpg,0.553860,0.723882,-0.046904,0.562730,0.631248,-0.077659,0.557694,0.660566,-0.043205,...,0.614849,0.465779,0.033657,0.600035,0.488247,0.033657,0.613536,0.514463,0.033657,0
3,angry\001_camA_S1_angry_00003.jpg,0.553998,0.721585,-0.046567,0.564419,0.633304,-0.078029,0.559075,0.662226,-0.043438,...,0.614056,0.465975,0.034289,0.598645,0.486928,0.034289,0.612574,0.512413,0.034289,0
4,angry\001_camA_S1_angry_00004.jpg,0.551908,0.720639,-0.046462,0.561848,0.629864,-0.076887,0.557130,0.659746,-0.042950,...,0.614649,0.464466,0.032677,0.599360,0.486611,0.032677,0.613911,0.511964,0.032677,0


# Xuất báo cáo thống kê, biểu đồ phân bố nhãn

In [15]:
for idx, label in enumerate(labels):
    print(f"Số lượng bỏ qua của {label} là {scan_fail[idx]}")
for idx, label in enumerate(labels):
    print(f"Số lượng quét được của {label} là {scan_success[idx]}")


Số lượng bỏ qua của angry là 0
Số lượng bỏ qua của happy là 0
Số lượng bỏ qua của neutral là 0
Số lượng bỏ qua của sad là 0
Số lượng bỏ qua của surprise là 2
Số lượng quét được của angry là 100
Số lượng quét được của happy là 100
Số lượng quét được của neutral là 100
Số lượng quét được của sad là 100
Số lượng quét được của surprise là 98


In [16]:
plt.figure(figsize=(10,6))

plt.bar(labels, scan_fail, label='Fail',color="navy", alpha=1)
plt.bar(labels, scan_success, label='Success',color="blue", alpha=0.2)

plt.xlabel('Emotion')
plt.ylabel('Count')
plt.title('Bar Chart')
plt.legend()
bar1 = plt.bar(labels, scan_fail, alpha=0)
bar2 = plt.bar(labels, scan_success, alpha=0)
plt.tight_layout()
plt.bar_label(bar1, padding=3)
plt.bar_label(bar2, padding=3)
plt.savefig('bar_chart.png', dpi=300)
plt.close()